In [ ]:
# Training Parameters
NUM_EPOCHS = 25
LEARNING_RATE = 1e-3
BATCH_SIZE = 32
DROPOUT_RATE = 0.25

# Dataset Parameters
SAMPLE_SIZE = 113000
IMAGE_SIZE = 224

print("Hyperparameters defined.")
print(f'Epochs: {NUM_EPOCHS}')
print(f'Learning Rate: {LEARNING_RATE}')
print(f'Batch Size: {BATCH_SIZE}')
print(f'Dropout Rate: {DROPOUT_RATE}')

In [ ]:
import os
import pandas as pd

DATA_ROOT = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"

print("DATA ROOT:", DATA_ROOT)
print("\nTop-level folders:")
print(os.listdir(DATA_ROOT))

csv_path = os.path.join(DATA_ROOT, "Data_Entry_2017.csv")

print("\nExpected CSV path:", csv_path)

if not os.path.exists(csv_path):
    raise FileNotFoundError(
        "Data_Entry_2017.csv not found. Check dataset version or path."
    )

df = pd.read_csv(csv_path)

print("\nLoaded CSV shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

In [ ]:
import os
import numpy as np

print("Indexing images... (this may take ~10-20s)")

image_paths = {}

for folder in os.listdir(DATA_ROOT):
    if folder.startswith("images"):
        folder_path = os.path.join(DATA_ROOT, folder)

        inner = os.path.join(folder_path, "images")

        if os.path.exists(inner):
            for file in os.listdir(inner):
                image_paths[file] = os.path.join(inner, file)

print("Total images indexed:", len(image_paths))

df["path"] = df["Image Index"].map(image_paths)

# Drop missing images
df = df[df["path"].notnull()].reset_index(drop=True)

print("Usable samples after mapping:", len(df))

# Using the SAMPLE_SIZE hyperparameter from BLOCK 0
if len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

print("Final training subset size:", len(df))

print("\nSample rows:")
print(df[["Image Index", "Finding Labels", "path"]].head())

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, label_to_idx, feature_columns, transform=None):
        self.df = dataframe
        self.label_to_idx = label_to_idx
        self.feature_columns = feature_columns
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def encode_labels(self, label_str):

        vec = np.zeros(len(self.label_to_idx), dtype=np.float32)

        for label in label_str.split("|"):
            if label in self.label_to_idx:
                vec[self.label_to_idx[label]] = 1.0

        return vec

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # image
        img = Image.open(row["path"]).convert("L")

        if self.transform:
            img = self.transform(img)

        # labels
        label = torch.tensor(
            self.encode_labels(row["Finding Labels"]),
            dtype=torch.float32
        )

        # extra features
        extra_features = torch.tensor(
            row[self.feature_columns].values.astype(np.float32),
            dtype=torch.float32
        )

        return img, extra_features, label

all_labels = sorted(
    set(
        label
        for labels in df["Finding Labels"]
        for label in labels.split("|")
    )
)

# Remove "No Finding" as standalone class behavior
if "No Finding" in all_labels:
    all_labels.remove("No Finding")

label_to_idx = {label: i for i, label in enumerate(all_labels)}

print("Classes:", all_labels)
print("Num classes:", len(all_labels))

# Patient Age: Scaling
scaler_age = StandardScaler()
df['Patient Age (scaled)'] = scaler_age.fit_transform(df[['Patient Age']])

# Patient Gender: One-hot encoding
df = pd.get_dummies(df, columns=['Patient Gender'], prefix='Gender', drop_first=False)
# Ensure both 'Gender_F' and 'Gender_M' exist for consistency, even if one is missing in subset
if 'Gender_F' not in df.columns: df['Gender_F'] = 0
if 'Gender_M' not in df.columns: df['Gender_M'] = 0


# View Position: One-hot encoding
df = pd.get_dummies(df, columns=['View Position'], prefix='ViewPos', drop_first=False)
# Ensure both 'ViewPos_AP' and 'ViewPos_PA' exist
if 'ViewPos_AP' not in df.columns: df['ViewPos_AP'] = 0
if 'ViewPos_PA' not in df.columns: df['ViewPos_PA'] = 0


# Define the list of new feature columns
feature_columns = [
    'Patient Age (scaled)',
    'Gender_F', 'Gender_M',
    'ViewPos_AP', 'ViewPos_PA'
]
# Ensure the columns actually exist in the dataframe after get_dummies.
feature_columns = [col for col in feature_columns if col in df.columns]

num_extra_features = len(feature_columns)
print(f"\nNumber of extra features: {num_extra_features}")
print(f"Extra feature columns: {feature_columns}")

# Use the IMAGE_SIZE hyperparameter from BLOCK 0
IMG_SIZE = IMAGE_SIZE

print("\nCalculating dataset mean and standard deviation for normalization...")

# Temporarily define a basic transform just to convert images to tensors
temp_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

# Instantiate a temporary dataset with the basic transform
temp_dataset = ChestXrayDataset(df, label_to_idx, feature_columns, temp_transform)

# Instantiate a temporary DataLoader to iterate through images
temp_loader = DataLoader(
    temp_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

mean = 0.
std = 0.
nb_samples = 0.

for images, _, _ in temp_loader:
    batch_samples = images.size(0)
    images = images.view(batch_samples, images.size(1), -1)
    mean += images.mean(2).sum(0)
    std += images.std(2).sum(0)
    nb_samples += batch_samples

mean /= nb_samples
std /= nb_samples

print(f"Calculated Mean: {mean.item():.4f}")
print(f"Calculated Std:  {std.item():.4f}")

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean.tolist(), std=std.tolist()) # Use calculated mean and std
])

dataset = ChestXrayDataset(df, label_to_idx, feature_columns, transform)

img, features, label = dataset[0]

print("\nImage shape:", img.shape)
print("Extra features:", features)
print("Extra features shape:", features.shape)
print("Label vector:", label)

In [ ]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

df["patient_id"] = df["Image Index"].apply(lambda x: x.split("_")[0])

print("Unique patients:", df["patient_id"].nunique())

splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)

train_idx, val_idx = next(
    splitter.split(df, groups=df["patient_id"])
)

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

print("Train patients:", train_df["patient_id"].nunique())
print("Val patients:", val_df["patient_id"].nunique())

print("Train samples:", len(train_df))
print("Val samples:", len(val_df))

train_dataset = ChestXrayDataset(train_df, label_to_idx, feature_columns, transform)
val_dataset = ChestXrayDataset(val_df, label_to_idx, feature_columns, transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Patient-level DataLoaders ready")

#class imbalance
num_classes = len(label_to_idx)

pos_counts = np.zeros(num_classes)

for labels in train_df["Finding Labels"]:
    for l in labels.split("|"):
        if l in label_to_idx:
            pos_counts[label_to_idx[l]] += 1

neg_counts = len(train_df) - pos_counts

pos_weights = neg_counts / (pos_counts + 1e-6)

pos_weights = torch.tensor(pos_weights, dtype=torch.float32).to(device)

print("Class weights computed")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SmallChestCNN(nn.Module):
    def __init__(self, num_classes, extra_features_dim, dropout_rate=0.25):
        super(SmallChestCNN, self).__init__()

        self.features = nn.Sequential(
            # First Block (Input: 1, 128, 128)
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # Output: 32, 64, 64
            nn.Dropout(p=dropout_rate),

            # Second Block (Input: 32, 64, 64)
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # Output: 64, 32, 32
            nn.Dropout(p=dropout_rate),

            # Third Block (Input: 64, 32, 32)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # Output: 128, 16, 16
            nn.Dropout(p=dropout_rate),

            # Fourth Block (Input: 128, 16, 16)
            nn.Conv2d(128, 256, kernel_size=3, padding=1), # Increased channels
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1))      # global pooling -> output 256 features
        )

        self.extra_features_mlp = nn.Sequential(
            nn.Linear(extra_features_dim, 32),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(32, 32)
        )

        self.classifier = nn.Linear(256 + 32, num_classes) # Adjusted input size

    def forward(self, x_img, x_extra): # Accept both image and extra features
        x_img = self.features(x_img)
        x_img = torch.flatten(x_img, 1) # Flatten image features (256)

        x_extra = self.extra_features_mlp(x_extra) # Process extra features (32)

        # Concatenate image features and extra features
        x_combined = torch.cat((x_img, x_extra), dim=1)

        x = self.classifier(x_combined)
        return x  # logits (NO sigmoid)

# Use the num_extra_features and DROPOUT_RATE defined in BLOCK 3 and BLOCK 0 respectively
model = SmallChestCNN(num_classes=len(label_to_idx), extra_features_dim=num_extra_features, dropout_rate=DROPOUT_RATE)
model = model.to(device)

print("Model initialized")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE) # Use LEARNING_RATE hyperparameter

print("Loss + optimizer ready")

In [ ]:
import torch
from sklearn.metrics import roc_auc_score
import numpy as np
import time

def evaluate(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    all_targets = []
    all_outputs = []

    with torch.no_grad():
        for images, extra_features, labels in loader: # Unpack extra_features
            images = images.to(device)
            extra_features = extra_features.to(device)
            labels = labels.to(device)

            outputs = model(images, extra_features) # Pass both to model
            loss = criterion(outputs, labels)

            total_loss += loss.item()

            all_targets.append(labels.cpu().numpy())
            all_outputs.append(torch.sigmoid(outputs).cpu().numpy())

    all_targets = np.vstack(all_targets)
    all_outputs = np.vstack(all_outputs)

    avg_loss = total_loss / len(loader)

    macro_auc = None
    per_class_aucs_list = [np.nan] * all_targets.shape[1] # Initialize with nan for all classes

    # Only attempt AUC calculation if there's at least one positive and one negative sample across all classes
    if all_targets.size > 0 and len(np.unique(all_targets)) > 1:
        try:
            # Calculate macro AUC
            macro_auc = roc_auc_score(all_targets, all_outputs, average="macro")

            # Calculate per-class AUCs
            # roc_auc_score with average=None returns AUC for each class.
            # It handles cases where a class might have only one unique label by returning NaN for that class.
            per_class_aucs_array = roc_auc_score(all_targets, all_outputs, average=None)
            per_class_aucs_list = per_class_aucs_array.tolist()

        except ValueError:
            # This catches cases where roc_auc_score cannot be computed, e.g., if a subset has no positive samples.
            # macro_auc remains None and per_class_aucs_list remains initialized with NaNs.
            pass

    return avg_loss, macro_auc, per_class_aucs_list # Return macro_auc and list of per-class AUCs

# Use the NUM_EPOCHS hyperparameter from BLOCK 0
EPOCHS = NUM_EPOCHS

train_losses = []
val_losses = []
val_aucs = []

print("Starting training...\n")

import os
import torch

SAVE_DIR = "/kaggle/working/checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

best_auc = -1

import pandas as pd

metrics_history = []

scaler = torch.amp.GradScaler('cuda')

for epoch in range(EPOCHS):
    epoch_start_time = time.time()
    model.train()

    running_loss = 0.0

    for images, extra_features, labels in train_loader:
        images = images.to(device)
        extra_features = extra_features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            outputs = model(images, extra_features) # Pass both to model
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    val_loss, val_auc, per_class_aucs = evaluate(model, val_loader, criterion) # Get per-class AUCs

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_aucs.append(val_auc)

    # Prepare metrics for current epoch
    current_metrics = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_auc": val_auc # This is the macro AUC
    }
    # Add per-class AUCs as individual columns
    for i, label_name in enumerate(all_labels):
        current_metrics[f"{label_name}"] = per_class_aucs[i]

    metrics_history.append(current_metrics)

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_auc": val_auc,
        "per_class_aucs": per_class_aucs, # Add per-class AUCs to checkpoint (as a list)
        "label_to_idx": label_to_idx,
        "feature_columns": feature_columns # Add feature_columns to checkpoint
    }

    # Save last epoch
    torch.save(checkpoint, f"{SAVE_DIR}/last_epoch.pt")

    # Save best model
    if val_auc is not None and val_auc > best_auc:
        best_auc = val_auc
        torch.save(checkpoint, f"{SAVE_DIR}/best_model.pt")
        print("Saved best model")

    epoch_end_time = time.time() # End timer for the epoch
    epoch_time_taken = epoch_end_time - epoch_start_time

    print(f"Epoch [{epoch+1}/{EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Val AUC:    {val_auc}")
    print(f"Time taken: {epoch_time_taken:.2f}s")
    print("-" * 30)


metrics_df = pd.DataFrame(metrics_history)

metrics_path = "/kaggle/working/metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

print("Saved metrics to:", metrics_path)
metrics_df


print("Training complete.")